# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ARTI-INTEL/fr-ml-intern-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring**

I chose this lane because it is the most **actionable** output for a content team: a ranked queue of pages to review, each with a suggested action and reason codes a human editor can inspect and override. The starter pipeline already runs end-to-end on this lane, which means I have a working baseline and real numbers to build on from day one.

Why this lane specifically:
- The dataset ships with a clear proxy label (`is_declining_label` from `trend_direction == "down"`) — so I can start measuring success immediately.
- The output is a **decision-support tool**, not a black-box score. Each recommendation includes reason codes like "stale visible page" or "low CTR visible page" that an editor can act on.
- The full warehouse (78.8M daily rows) lets me upgrade from the starter's current-window proxy label to a **future-looking label** like "declines in the next 30 days" — which is a stronger, leakage-audited target.
- There is a clear baseline (hand-written rule scoring 0.240 Precision@50) to beat, and the starter already proves a learned model can roughly **triple** that (0.740 Precision@50).

In [ ]:
# Code placeholder for section 1 — verifying the lane data exists
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Starter dataset: {len(df):,} rows, {df['client_id'].nunique()} clients")
print(f"Content types available: {df['content_type'].unique().tolist()}")

## 2. The question: decision, action, cost of a wrong call

### Search question

> **Which pages should a content team review first for refresh, expansion, or pruning?**

### Unit of analysis

**One row = one content item (one page)** — the grain of `dim_content` and the starter dataset. Each page gets a score, a suggested action, and a list of reason codes explaining why it scored that way.

### The decision this improves

A content operations manager allocates limited editor hours (maybe 20-50 pages per week) to review and improve existing content. The question is **which pages to put in that queue**. Currently they might use gut feel, first-in-first-out, or simple rules (e.g. "pages older than 6 months"). This project ranks pages by evidence-based opportunity instead.

### The action someone could take

| Score tier | Action for the editor |
|---|---|
| High confidence (top 20%) | Review the page. Rewrite title/meta, expand thin content, or update stale information. |
| Medium confidence | Quick inspection. Prioritize below high-confidence items. |
| Low confidence / monitor | No immediate action. Track over next review cycle. |

### Cost of a wrong recommendation

Two kinds of wrong:

| Wrong type | What happens | Cost |
|---|---|---|
| **False positive** (recommend a page that doesn't need work) | Editor wastes 15-30 minutes reviewing a healthy page | Lost editor time — manageable at low volume, costly at scale |
| **False negative** (miss a page that really needs work) | A declining page keeps losing traffic, position, and revenue — and becomes harder to recover the longer it sits | Harder to measure, but likely **far larger**: missed opportunity × compounding decay |

This is why **Precision@K** is the right primary metric — top-K captures the queue capacity, and precision minimizes wasted editor time. Average precision captures ranking quality across the full list.

### Why data or ML can help at all

Content decay is **not random**. Pages with certain patterns (aging content, dropping position, thin word count, low CTR for their position tier) are systematically more likely to decline. A hand-written rule can catch some of these (the baseline catches about 12 of the top 50), but the interactions are complex:

- A **stale page in position 1-3** needs a different treatment than a **stale page at position 20+**
- **Low CTR** means different things at position 3 (something is wrong with the snippet) vs position 10 (the page might be ranking for the wrong query)
- The relationship between content length and performance is **not linear** — some short pages perform fine

A learned model can pick up these interactions in ways a fixed formula cannot. The starter data confirms this: the random forest (0.740 Precision@50) roughly **triples** the baseline rule (0.240 Precision@50).

In [ ]:
# Verifying the output structure exists — run the pipeline first if this fails
import json, sys
try:
    results = json.load(open("outputs/model_results.json"))
    print(f"Best model selected by Precision@50: {results['best_model']}")
    print(f"Split strategy: {results['split_strategy']}")

    base_p50 = results['baseline']['baseline_precision_at_50']
    rf_p50 = results['models']['random_forest']['precision_at_50']
    print(f"\nBaseline  Precision@50: {base_p50:.3f}  ({round(base_p50 * 50)} of top 50 right)")
    print(f"RF model  Precision@50: {rf_p50:.3f}  ({round(rf_p50 * 50)} of top 50 right)")
    print(f"\nImprovement: {rf_p50 / base_p50:.1f}x — ML beats the hand-written rule decisively.")
except FileNotFoundError:
    print("outputs/model_results.json not found.")
    print("Run the starter pipeline first:  python scripts/run_all.py")
    sys.exit(1)

## 3. Quick look at the data (2-3 real numbers)

Every number below is **computed live** from `data/raw/content_refresh_anonymized.csv` and `outputs/model_results.json` — nothing is hardcoded.

In [ ]:
import pandas as pd
import json

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("=" * 65)
print("NUMBER 1 — The scale of the content decay problem")
print("=" * 65)
total = len(df)
declining = (df['trend_direction'] == 'down').sum()
declining_rate = declining / total * 100
print(f"Out of {total:,} pages across {df['client_id'].nunique()} clients:")
print(f"  -> {declining:,} pages ({declining_rate:.1f}%) are currently declining")
print(f"  -> {total - declining:,} pages are stable, flat, up, or new")
print()
print("Interpretation: More than half the content inventory is losing visibility.")
print("Even if only the top 5% can be reviewed per cycle, that's")
print(f"~{declining // 20:,} high-priority candidates to work through.")
print()

print("=" * 65)
print("NUMBER 2 — Most pages drastically underperform their potential")
print("=" * 65)
med_imp = df['impressions_90d'].median()
med_clicks = df['clicks_90d'].median()
med_ctr = df['ctr'].median()
print(f"Median page stats over 90 days:")
print(f"  -> {med_imp:,.0f} impressions (some have up to {df['impressions_90d'].max():,})")
print(f"  -> {med_clicks:,.0f} click (most pages get essentially ZERO clicks)")
print(f"  -> Median CTR: {med_ctr:.2f}% (x100 scale in data)")
print()

# Pages with high impressions but near-zero CTR
high_imp_low_ctr = df[
    (df['impressions_90d'] >= 500) &
    (df['ctr'] < 0.5) &
    (df['avg_position'] > 0) &
    (df['avg_position'] <= 20)
]
print(f"Pages with >=500 impressions AND avg position 1-20 AND CTR < 0.5%:")
print(f"  -> {len(high_imp_low_ctr):,} pages ({len(high_imp_low_ctr)/total*100:.1f}% of inventory)")
print(f"  -> These are visible pages that should be getting clicks, but aren't.")
print(f"  -> Each one is a CTR-optimization candidate (title tag, meta description).")
print()

print("=" * 65)
print("NUMBER 3 — The model already proves this lane works")
print("=" * 65)
res = json.load(open("outputs/model_results.json"))
base = res['baseline']['baseline_precision_at_50']
rf = res['models']['random_forest']['precision_at_50']
print(f"Baseline rule   -> Precision@50 = {base:.3f}  ({round(base * 50)}/{50} correct)")
print(f"Random Forest   -> Precision@50 = {rf:.3f}  ({round(rf * 50)}/{50} correct)")
print(f"Improvement: {rf/base:.1f}x -- the learned model triples the rule's accuracy.")
print()
print("This is on the 30K-row starter slice with a simple proxy label.")
print("With the full 78.8M-row warehouse, we can define a stronger future-looking label")
print("(e.g. 'will this page decline in the NEXT 30 days?') and potentially do even better.")

### Summary: Why this lane is worth 7 weeks

| Evidence | What it tells us |
|---|---|
| **54.2%** of pages are declining | The problem is large — this isn't edge-case tuning |
| **3.1x improvement** over the baseline rule | There is real signal in the data that a simple formula misses |
| **~6,500+ pages** with high impressions + low CTR + good position | Specific, actionable opportunity segments exist — pages editors can fix today |
| 78.8M-row warehouse available in week 3 | Room to define future-looking labels, stronger validation, and per-client personalization |

In [ ]:
# Optional: Preview the final refresh queue (run the pipeline first if this fails)
import pandas as pd, os
queue_path = "outputs/refresh_queue.csv"
sample_path = "outputs/refresh_queue_sample.csv"

if os.path.exists(queue_path):
    queue = pd.read_csv(queue_path)
elif os.path.exists(sample_path):
    queue = pd.read_csv(sample_path)
    print("(Using sample — run 'python scripts/run_all.py' for the full queue)\n")
else:
    print("Queue file not found. Run 'python scripts/run_all.py' first to generate outputs.")
    queue = None

if queue is not None:
    print("Final refresh queue sample (top 5 rows):")
    cols = ['final_rank', 'final_refresh_score', 'confidence', 'suggested_action',
            'is_declining_label', 'impressions_90d', 'ctr']
    # only show columns that exist
    cols = [c for c in cols if c in queue.columns]
    print(queue.head()[cols].to_string(index=False))

## 4. Careful words: what I can and can't claim

### What this work CAN say (safe, directional claims)

- **Observed**: "Pages with certain patterns (e.g. dropping position, low CTR for their tier, stale content) are more likely to show declining traffic."
- **Directional**: "The top-ranked pages in our model are estimated to be XX% more likely to need review than a random page."
- **Decision-support**: "Using this ranked queue to prioritize review could save an estimated YY editor-hours per cycle while catching more at-risk pages."
- **Measured**: "On the starter dataset, the model achieved Precision@50 of 0.74, meaning 37 of the top 50 recommendations were declining pages."

### What this work will NEVER claim

| Claim | Why we can't say it |
|---|---|
| "Refreshing this page will cause it to recover" | Correlation is not causation. We rank candidates; we don't run experiments. |
| "We predicted Google's algorithm" | We only observe search signals. We don't know Google's ranking logic. |
| "This page has a priority_score of X" (FlyRank's product score) | Product scores (`health_score`, `priority_score`) are deliberately NOT in our data. |
| "We know the real URL/keyword/query/content" | All identifiers are pseudonymized. We work with hashes only. |
| "AI referral pattern proves AI citation/ranking" | AI session data only measures click-throughs, not citations or rankings. |

### The honest framing

This project builds a **decision-support ranking system** — it tells a content editor "here are the pages most likely worth your time, in order, with reasons." It does not tell them "refresh this and traffic will grow." That second claim requires a controlled experiment, which is outside this project's scope.

The value is in **focusing limited human attention** on the pages where evidence of opportunity is strongest. Even a 2x improvement in editor productivity (fewer wasted reviews, more impactful fixes) would be a practical win.

In [ ]:
# Verify the claims we can make are backed by numbers
print("--- Claims audit: verifying what the data actually supports ---\n")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"1. Declining rate: {((df['trend_direction']=='down').mean()*100):.1f}%")
print("   -> DIRECTLY OBSERVED from trend_direction column")

import json
res = json.load(open("outputs/model_results.json"))
print(f"2. Model Precision@50: {res['models']['random_forest']['precision_at_50']:.3f}")
print("   -> MEASURED from client-holdout evaluation")

high_imp_low_ctr = len(df[
    (df['impressions_90d'] >= 500) &
    (df['ctr'] < 0.5) &
    (df['avg_position'] > 0) &
    (df['avg_position'] <= 20)
])
print(f"3. High-impression/low-CTR pages: {high_imp_low_ctr:,}")
print("   -> COUNTED from the data")
print()
print("All numbers are reproducible from the shipped starter dataset.")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.